In [17]:
import sys
import os

sibling_dir = os.path.abspath('../01_agentic_rag')

if sibling_dir not in sys.path:
    sys.path.append(sibling_dir)

In [18]:
from dotenv import load_dotenv
load_dotenv()

# configure the OpenAI client with your API key
from openai import OpenAI

openai_client = OpenAI()

In [19]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [20]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [21]:
query = "How do I run Kafka?"

query_vector = model.encode(query)
results = vs_index.search(query_vector, num_results=5, filter_dict={"course": "llm-zoomcamp"})
results

[{'id': '1d0b969028',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Ollama: How to install Ollama?',
  'answer': 'First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{"models": [...]}  \n```\n\nThen, install the Python client with:\n\n```bash\npip install ollama\n```\n\n

In [22]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [23]:
# we subclass RAGBase and override search to encode the query before searching


class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )


In [24]:
vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client,
)

In [25]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes, you can still join. The course is open, and you can start learning and submitting homework while the form is open. If you want a certificate, make sure to submit your project before submissions close.'

In [26]:
vs_index.close()